# valorant_minimap YOLOv8 Training

**実行前に:** ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択

セルを上から順に実行してください。最後に `valorant_minimap.onnx` がダウンロードされます。

## Step 1: 依存パッケージのインストール

In [ ]:
!pip install ultralytics==8.2.87 onnx==1.16.1 onnxsim==0.4.36 onnxruntime==1.18.1 roboflow==1.1.46 --quiet
print('インストール完了')

## Step 2: Roboflow からデータセットをダウンロード

**API キーを下のセルに入力してください。**

In [ ]:
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY = ''  # ← ここに API キーを入力

if not ROBOFLOW_API_KEY:
    raise ValueError('ROBOFLOW_API_KEY を入力してください')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace('ddsssers-workspace') \
             .project('valorant-minimap-de8vw') \
             .version(1) \
             .download('yolov8', location='/content/data/minimap')

print('ダウンロード完了')
!find /content/data/minimap -name '*.yaml' | head -3
!find /content/data/minimap/train/labels -name '*.txt' | wc -l

## Step 3: dataset.yaml を確認・修正

Roboflow がダウンロードした yaml のクラス順序が正しいか確認します。

**必須:** クラス順序は `['minimap_region', 'player_dot', 'enemy_dot']` でなければなりません。

In [ ]:
import glob, yaml

yaml_path = glob.glob('/content/data/minimap/*.yaml')[0]
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

print('現在のクラス:', cfg.get('names'))

REQUIRED = ['minimap_region', 'player_dot', 'enemy_dot']
if cfg.get('names') != REQUIRED:
    print('⚠ クラス順序が異なります。修正します...')
    cfg['names'] = REQUIRED
    cfg['nc'] = len(REQUIRED)
    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f)
    print('修正後:', cfg['names'])
else:
    print('✓ クラス順序は正しいです')

print('\ndataset.yaml 全内容:')
print(yaml.dump(cfg))

## Step 4: 学習

T4 GPU で約 1 時間かかります。

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    task='detect',
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name='valorant_minimap',
    project='/content/runs',
    fliplr=0.0,
    hsv_v=0.3,
    hsv_s=0.4,
    mosaic=0.5,
    copy_paste=0.1,
    iou=0.5,
    conf=0.45,
)

## Step 5: 精度確認

player_dot クラスの mAP@0.5 が **0.70 以上**であることを確認します。
未達の場合は下のセルで再学習オプションを確認してください。

In [ ]:
import pandas as pd
from pathlib import Path

results_csv = Path('/content/runs/valorant_minimap/results.csv')
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

# 最終エポックの mAP@0.5
last = df.iloc[-1]
map50 = last.get('metrics/mAP50(B)', last.get('val/mAP50', None))
print(f'最終 mAP@0.5: {map50:.4f}')

if map50 is not None and map50 >= 0.70:
    print('✓ 精度目標達成 — エクスポートに進みます')
else:
    print('⚠ 精度目標未達 (0.70 必要)')
    print('対策: 追加データを収集するか、下のセルで yolov8s に切り替えて再学習')

In [ ]:
# 精度未達の場合のみ実行 — yolov8s で再学習
# 通常のセルでは実行しないでください

# model_s = YOLO('yolov8s.pt')
# model_s.train(
#     task='detect', data=yaml_path, epochs=100, imgsz=640, batch=8,
#     name='valorant_minimap_s', project='/content/runs',
#     fliplr=0.0, hsv_v=0.3, iou=0.5, conf=0.45,
# )

## Step 6: ONNX エクスポート + レイアウト修正

YOLOv8 の ONNX 出力は `[1, 7, 8400]` チャネル優先。
アプリの `parseDetections()` が期待する `[8400, 7]` 行優先に変換します。

In [ ]:
import numpy as np
import onnx
import onnxsim
from onnx import helper, TensorProto, numpy_helper
from pathlib import Path
from ultralytics import YOLO

NC = 3  # minimap_region, player_dot, enemy_dot
best_pt = Path('/content/runs/valorant_minimap/weights/best.pt')

# --- Export ---
model = YOLO(str(best_pt))
model.export(format='onnx', imgsz=640, opset=11, simplify=True, dynamic=False)

exported = best_pt.with_suffix('.onnx')
proto = onnx.load(str(exported))

# --- Assertions ---
assert 'images' in [i.name for i in proto.graph.input], 'input node name mismatch'
in_shape = [d.dim_value for d in proto.graph.input[0].type.tensor_type.shape.dim]
assert in_shape == [1, 3, 640, 640], f'input shape: {in_shape}'
raw_out = [d.dim_value for d in proto.graph.output[0].type.tensor_type.shape.dim]
assert raw_out == [1, 4 + NC, 8400], f'raw output: {raw_out}'
print(f'[OK] input={in_shape}, raw output={raw_out}')

# --- Transpose [1, 7, 8400] → [1, 8400, 7] ---
out_name = proto.graph.output[0].name
transposed = out_name + '_T'
proto.graph.node.append(
    helper.make_node('Transpose', [out_name], [transposed], perm=[0, 2, 1]))

# --- Reshape [1, 8400, 7] → [8400, 7] ---
shape_init = numpy_helper.from_array(
    np.array([8400, 4 + NC], dtype=np.int64), 'reshape_shape')
proto.graph.initializer.append(shape_init)
reshaped = out_name + '_R'
proto.graph.node.append(
    helper.make_node('Reshape', [transposed, 'reshape_shape'], [reshaped]))

new_out = helper.make_tensor_value_info(reshaped, TensorProto.FLOAT, [8400, 4 + NC])
del proto.graph.output[:]
proto.graph.output.append(new_out)

# --- Re-simplify ---
proto, ok = onnxsim.simplify(proto)
assert ok, 'onnxsim failed'

final_shape = [d.dim_value for d in proto.graph.output[0].type.tensor_type.shape.dim]
assert final_shape == [8400, 4 + NC], f'final shape: {final_shape}'
print(f'[OK] final output shape = {final_shape}')

# --- Save ---
out_path = Path('/content/valorant_minimap.onnx')
onnx.save(proto, str(out_path))
print(f'[SAVED] {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)')

## Step 7: 検証

In [ ]:
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession('/content/valorant_minimap.onnx')

assert sess.get_inputs()[0].name == 'images'
assert list(sess.get_inputs()[0].shape) == [1, 3, 640, 640]

dummy = np.full((1, 3, 640, 640), 0.45, dtype=np.float32)
out = sess.run(None, {'images': dummy})[0]

assert out.shape == (8400, 7), f'shape error: {out.shape}'

above_conf = int(np.sum(np.max(out[:, 4:], axis=1) >= 0.45))
print(f'[OK] output shape = {out.shape}')
print(f'[OK] grey frame detections (expect ~0): {above_conf}')
print('\n✓ 全チェック通過 — 次のセルでダウンロードしてください')

## Step 8: ONNX ファイルをダウンロード

ダウンロードしたファイルを `src-tauri/resources/models/valorant_minimap.onnx` に配置してください。

In [ ]:
from google.colab import files
files.download('/content/valorant_minimap.onnx')
print('ダウンロード開始 — ブラウザの Downloads フォルダを確認してください')